# **Installation**

In [2]:
!pip install catboost -q

# **Import Libraries**

In [3]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, make_scorer, confusion_matrix)
from imblearn.metrics import geometric_mean_score, specificity_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.stats import loguniform
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import ADASYN

warnings.filterwarnings('ignore')



# **Load Dataset**

In [4]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

# **Data Preprocessing**

In [5]:
 # Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)


# Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])


#  Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})


#  One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)


#  Split Features & Target
X = df.drop('y', axis=1)
y = df['y']

#  Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# **Scaling**

In [6]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# **Sampling Technique**

In [7]:
adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_s, y_train)

# **Models**

In [8]:
model_name = {

    "Logistic Regression": (
        LogisticRegression(solver='saga', max_iter=500),
         {
             "C": [0.1, 1, 100]

          }
        ),

    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
         {
            "max_depth": [5, 10, 12],

            "min_samples_split": [10, 20]

           }
        ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
         {
             "max_depth": [10, 20, None],
             "max_features": ['sqrt']
          }
        ),
    "Gradient Boosting": (
        GradientBoostingClassifier(n_estimators=100, random_state=42),
         {
             "learning_rate": [0.05, 0.1, 0.2],
             "max_depth": [3, 5]
          }
        ),
    "XGBoost": (
        XGBClassifier(n_estimators=100, tree_method='hist', n_jobs=-1, eval_metric='logloss'),
         {
             "learning_rate": [0.05, 0.1, 0.2],
             "max_depth": [4, 6]
             }
        ),
    "LightGBM": (
        LGBMClassifier(n_estimators=100, n_jobs=-1, verbose=-1),
         {
             "learning_rate": [0.05, 0.1],
             "num_leaves": [31, 63]
          }
        ),
    "CatBoost": (
        CatBoostClassifier(iterations=100, thread_count=-1, verbose=0),
         {
             "depth": [4, 6],
             "learning_rate": [0.05, 0.1]
          }
        ),
    "SVM": (
    CalibratedClassifierCV(LinearSVC(max_iter=2000, dual=False)),
    { "estimator__C": [0.1, 1, 10] }
),
    "KNN": (
        KNeighborsClassifier(n_jobs=-1),
         {
             "n_neighbors": [3, 5, 7],
             "weights": ['uniform', 'distance']
          }
        ),
    "MLP": (
        MLPClassifier(max_iter=200, early_stopping=True),
         {
             "hidden_layer_sizes": [(64,),(64,32)],
            "alpha": [0.0001, 0.01]
             }
        ),
    "Bagging": (
        BaggingClassifier(n_estimators=50, n_jobs=-1),
         {
             "max_samples": [0.7,0.9],
             "max_features": [0.7, 1.0]
          }
        )


}


# **For storing**

In [9]:
results= []

# **Matrics Function**

In [10]:
def compute_metrics(y_true, y_pred, y_prob, name_of_model):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    acc_val = accuracy_score(y_true, y_pred)
    prec_val = precision_score(y_true, y_pred, zero_division=0)
    rec_val = recall_score(y_true, y_pred, zero_division=0)
    spec_val = tn / (tn + fp) if (tn + fp) != 0 else 0
    f1_val = f1_score(y_true, y_pred, zero_division=0)
    roc_val = roc_auc_score(y_true, y_prob)
    mcc_val = matthews_corrcoef(y_true, y_pred)
    gmean_val = geometric_mean_score(y_true, y_pred)
    kappa_val = cohen_kappa_score(y_true, y_pred)


    print(f"Accuracy    : {acc_val:.4f}")
    print(f"Precision   : {prec_val:.4f}")
    print(f"Recall      : {rec_val:.4f}")
    print(f"Specificity : {spec_val:.4f}")
    print(f"F1 Score    : {f1_val:.4f}")
    print(f"ROC-AUC     : {roc_val:.4f}")
    print(f"MCC         : {mcc_val:.4f}")
    print(f"G-Mean      : {gmean_val:.4f}")
    print(f"Kappa       : {kappa_val:.4f}")

    me_dic = {
        "Model Name": name_of_model,
        "Accuracy": acc_val,
        "Precision": prec_val,
        "Recall": rec_val,
        "Specificity": spec_val,
        "F1 Score": f1_val,
        "ROC-AUC": roc_val,
        "MCC": mcc_val,
        "G-Mean": gmean_val,
        "Kappa": kappa_val
    }

    results.append(me_dic)


# **Cross Validation Setup**

In [11]:
skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# **Evaluation Metrics**

In [12]:
print("\n" + "="*40)
print("ADASYN ")
print("="*40)

best_models = {}

for name, (model_obj, params) in model_name.items():

    rscv = RandomizedSearchCV(
        estimator=model_obj,
        param_distributions=params,
        n_iter=5,
        scoring='roc_auc',
        cv=skf,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )

    rscv.fit(X_train_ad, y_train_ad)

    best_model = rscv.best_estimator_
    best_models[name] = best_model

    y_pred = best_model.predict(X_test_s)
    if hasattr(best_model, "predict_proba"):
        y_prob = best_model.predict_proba(X_test_s)[:, 1]
    else:
        y_prob = best_model.decision_function(X_test_s)

    print(f"\nModel: {name}")
    compute_metrics(y_test, y_pred, y_prob, name)
stacking_model = StackingClassifier(
  estimators=[
    ('rf',   best_models["Random Forest"]),
    ('xgb',  best_models["XGBoost"]),
    ('lr',   best_models["Logistic Regression"]),
    ('dt',   best_models["Decision Tree"])
    ],
final_estimator=RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
cv=3,
n_jobs=-1
)

stacking_model.fit(X_train_ad, y_train_ad)

y_pred_st = stacking_model.predict(X_test_s)
y_prob_st = stacking_model.predict_proba(X_test_s)[:, 1]

print(f"\nModel: Stacking")
compute_metrics(y_test, y_pred_st, y_prob_st, "Stacking")


ADASYN 

Model: Logistic Regression
Accuracy    : 0.7873
Precision   : 0.3062
Recall      : 0.7015
Specificity : 0.7982
F1 Score    : 0.4263
ROC-AUC     : 0.7979
MCC         : 0.3611
G-Mean      : 0.7483
Kappa       : 0.3196

Model: Decision Tree
Accuracy    : 0.8773
Precision   : 0.4479
Recall      : 0.3847
Specificity : 0.9398
F1 Score    : 0.4139
ROC-AUC     : 0.7220
MCC         : 0.3471
G-Mean      : 0.6013
Kappa       : 0.3458

Model: Random Forest
Accuracy    : 0.8864
Precision   : 0.4942
Recall      : 0.3664
Specificity : 0.9524
F1 Score    : 0.4208
ROC-AUC     : 0.7743
MCC         : 0.3643
G-Mean      : 0.5907
Kappa       : 0.3593

Model: Gradient Boosting
Accuracy    : 0.9009
Precision   : 0.6045
Recall      : 0.3491
Specificity : 0.9710
F1 Score    : 0.4426
ROC-AUC     : 0.8038
MCC         : 0.4104
G-Mean      : 0.5822
Kappa       : 0.3925

Model: XGBoost
Accuracy    : 0.9013
Precision   : 0.6148
Recall      : 0.3319
Specificity : 0.9736
F1 Score    : 0.4311
ROC-AUC     : 0.

In [13]:
results_df = pd.DataFrame(results)
results_df.to_excel("ADASYN_SAMPLING(Hyperparameter_tuning_RSCV)_Results.xlsx", index=False)